# Probing the word2vec neighbors

My SGNS vectors (seed 0) next to gensim's, the polysemy probe, and the subword (BPE-unit) model on misspellings.

In [1]:
# setup: run from the repository root (python notebooks/build.py)
import sys; sys.path.insert(0, '.')

In [2]:
from word2vec.model import Embeddings
mine = Embeddings.load('word2vec/artifacts/seed0')
gens = Embeddings.load('word2vec/artifacts/gensim_seed0')
bpe = Embeddings.load('word2vec/artifacts/bpe_seed0')
print(len(mine.words), 'words;', mine.meta['dim'], 'dims')

22948 words; 300 dims


## Finance probes: mine vs gensim

In [3]:
for w in ['garnish', 'escrow', 'forbearance', 'chargeback', 'furnisher', 'autopay']:
    print(f'{w:12s} mine:  ', [n for n, _ in mine.most_similar(w, 6)])
    print(f'{"":12s} gensim:', [n for n, _ in gens.most_similar(w, 6)])

garnish      mine:   ['wages', 'garnishing', 'garnished', 'garnishment', 'garnishments', 'wage']
             gensim: ['wages', 'garnishing', 'garnished', 'garnishments', 'garnishment', 'messerli']
escrow       mine:   ['shortage', 'analysis', 'taxes', 'escrowed', 'projected', 'escrowing']
             gensim: ['shortage', 'analysis', 'escrowed', 'shortages', 'taxes', 'shortfall']
forbearance  mine:   ['deferment', 'forebearance', 'deferral', 'forebarance', 'cares', 'deferement']
             gensim: ['forebearance', 'deferment', 'deferral', 'post-forbearance', 'forebarance', 'forberance']
chargeback   mine:   ['merchant', 'charge-back', 'rebilled', 'decisioned', 'acquirer', 'amx']
             gensim: ['merchant', 'charge-back', 'rebilled', 'rebill', 're-charge', 're-bill']
furnisher    mine:   ['623', 'furnishers', 'learns', 'reinvestigations', 'completeness', 'metro2']
             gensim: ['learns', 'furnishers', '623', 'metro2', 'reinvestigations', 'collectionspractices']
autopay 

## One vector per word: polysemy

In [4]:
for w in ['charge', 'balance', 'dispute', 'interest']:
    print(f'{w:9s}', [n for n, _ in mine.most_similar(w, 10)])

charge    ['charges', 'charged', 'recharged', 'rebill', 'dicover', 're-charged', 'reaccuring', 'comentiy', 'rebilling', 'transaction']
balance   ['updatedxxxx', '<AMT>', 'updatedxx', 'termsna', 'balnce', 'accountxxxx', 'amount', 'statusclosed', 'typesecured', 'statuspaid']
dispute   ['disputes', 'disputed', 'dipute', 're-dispute', 're-disputed', 'desputed', 'disputing', 'redispute', 'iteam', 'reinvestigating']
interest  ['prinicpal', 'accruing', 'accrues', 'proportionally', 'accrued', 'rate', 'interst', 'consol', 'accruements', 'capitalizes']


## Misspellings: word-level has no vector, BPE units do

In [5]:
for w in ['forbearnace', 'garnishmnet', 'reposession', 'delinqeunt']:
    v = bpe.vector(w)
    print(f'{w:13s} word-level vector: {w in mine.index}; BPE pieces: {bpe.tokenizer.pieces(" " + w)}')

forbearnace   word-level vector: False; BPE pieces: ['▁for', 'bear', 'n', 'ace']
garnishmnet   word-level vector: False; BPE pieces: ['▁garnish', 'm', 'net']
reposession   word-level vector: True; BPE pieces: ['▁repo', 's', 'ession']
delinqeunt    word-level vector: False; BPE pieces: ['▁de', 'lin', 'q', 'e', 'unt']


## Analogy-style arithmetic (for curiosity; not part of the evaluation)

In [6]:
import numpy as np
def analogy(a, b, c, k=5):
    v = mine.vector(b) - mine.vector(a) + mine.vector(c)
    sims = mine.unit @ (v / np.linalg.norm(v))
    return [mine.words[i] for i in np.argsort(-sims) if mine.words[i] not in (a, b, c)][:k]
print('mortgage : foreclosure :: car :', analogy('mortgage', 'foreclosure', 'car'))
print('equifax : experian :: transunion? ->', [n for n, _ in mine.most_similar('equifax', 5)])

mortgage : foreclosure :: car : ['vehicle', 'repossessed', 'repoed', 'repossession', 'lemon']
equifax : experian :: transunion? -> ['experian', 'transunion', 'exquifax', 'eqifax', 'innovis']
